# FORESEE Models: Heavy Neutral Leptons (HNLs)

## Load Libraries 

In [ ]:
import numpy as np
import sys,os
src_path = "../../../"
sys.path.append(src_path)
src_path = "../"
sys.path.append(src_path)
from src.foresee import Foresee, Utility, Model
from matplotlib import pyplot as plt
from HNLCalc import * 

## 1. Specifying the Model

Heavy Neutral Leptons, denoted by $N$, are fermionic gauge singlets that can be added as an extenstion to the standard model. These HNLs mix with the SM active neutrinos, and thus pick up couplings to the SM which are suppresed by a small mixing angle, $U_\alpha$ for $\alpha = e, \mu, \tau$. The phenomenology of these interactions can be described by the following Lagrangian: 
\begin{equation} 
\mathcal{L} \supset -{m_N} \bar{N}^c N  - \frac{1}{\sqrt{2}}g\sum_{\alpha=e,\mu,\tau} {U^*_\alpha}  W_\mu^+  \,\overline{N^c} \, \gamma^\mu l_\alpha  - \frac{1}{2\cos\theta_W}g\sum_{\alpha=e,\mu,\tau} {U^*_\alpha} Z_\mu  \overline{N^c}\, \gamma^\mu \nu_\alpha+ \text{h.c.} . 
\end{equation}

In principle, this model has four free parameters: the mass $m_N$ and the three mixing angles $U_\alpha$. In practice, however, we will therefore impose some relations between the mixing angles to reduce the number of free parameters. In particular, we define $U_\alpha = g \times V_\alpha$ with $V_\alpha= \hat{U}_\alpha$ being the normalized coupling vector, so $|V_\alpha|=1$. Using this definition, we have the HNL mass $m_N$ and the coupling parameter $g$ as free parameters. 

In [ ]:
energy = "13.6"
modelname="HNL-tau"
model = Model(modelname, path="./")

# Builder parameters, mirroring the build.py / load_model() inputs.
nsample_2body = 10
nsample_3body = 10
generators_light = ["EPOSLHC", "SIBYLL", "QGSJET"]
generators_heavy = ["NLO-P8", "NLO-P8-Max", "NLO-P8-Min"]
ve, vmu, vtau = 0, 0, 1


**Model Initialization:** The production and decay rates of the HNL depend not only on the choice of $m_N$ and $g$, but also $V_\alpha$. In addition, the number of production modes in sgificantly larger than in other models. To simplify the process of setting up the model we provide the class `HeavyNeutralLepton(ve, vmu, vtau)`. Here `ve`, `vmu` and `vtau` correspond to the three entries of $V_\alpha$. In the following case, we choose $V_e=1$ and $V_\mu=V_\tau=0$. We also define the generators for light/heavy hadrons. 

In [ ]:
hnl = HeavyNeutralLepton(ve=ve, vmu=vmu, vtau=vtau)
hnl.set_generators(
    generators_light = generators_light,
    generators_heavy = generators_heavy,
)


**Production:** The leptonic two body decay of a pseudoscalar meson $P \to l N$ is a dominant production mechanism for HNL's. In addition, HNLs with non-vanishing coupling to taus can be produced in two body tau decays $\tau \to M N$ with $M = \pi, K, \rho, K^*$. We can load all the non-vanishing channels using `hnl.get_channels_2body()`. 

Additionally, HNLs can be produced through three body pseudoscalar meson decay $P \to P'lN, VlN$ or through three body tau decays $\tau \to l \nu N$. The non-vanishing three body channels can likewise be loaded using `hnl.get_channels_3body()`. 

In [ ]:
production_channels = []
for label, pid0, pid1, br, generator, description in hnl.get_channels_2body():
    print ('include:', description)
    model.add_production_2bodydecay(
        label = label,
        pid0 = pid0,
        pid1 = pid1,
        br = br,
        generator = generator,
        energy = energy,
        nsample = nsample_2body,
    )
    production_channels.append([label, None, description])

for label, pid0, pid1, pid2,br,generator,integration,description in hnl.get_channels_3body():
    print ('include:', description)
    model.add_production_3bodydecay(
        label = label,
        pid0 = pid0,
        pid1 = pid1,
        pid2 = pid2,
        br = br,
        generator = generator,
        energy = energy,
        nsample = nsample_3body,
        integration = integration,
    )
    production_channels.append([label, None, description])

**Decay:** HNLs can decay into purely leptonic finalstates, $\nu\, l_\alpha^+ l_\beta^-$ or $\nu\nu\nu$, and Hadronic finalstates, $\nu H^0 (q\overline{q})$ or $l^\pm H^\mp (q\overline{q}')$,   with a decay width proportional to $g^2$. Since we are considering Majorana HNLs, LNV decays are included. We implemented the decay formulas given in [1905.00284](https://arxiv.org/abs/1905.00284). The function `hnl.get_br_and_ctau()` is used to generate the decay branching fractions and lifetimes and stores them in the `model/br` and `model/ctau` folders respectively. The function `hnl.set_brs()` provides the input needed to load the branching fractions into FORESEE. 

In [ ]:
hnl.get_br_and_ctau()

In [ ]:
model.set_ctau_1d(
    filename=f"model/ctau/ctau.txt",
    coupling_ref=1 
)

modes,finalstates,filenames = hnl.set_brs()

model.set_br_1d(
    modes=modes,
    finalstates=finalstates,
    filenames=filenames
)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path="../../../")
foresee.set_model(model=model)

## 2. Event Generation

In the following, we want to study one specific benchmark point with $m_{N}=1$ GeV and $\epsilon=1\cdot 10^{-3}$ and export events as a HEPMC file. 

In [ ]:
mass, coupling, = 1, 1e-3

First, we will produce the corresponding flux for this mass and a reference coupling $\epsilon_{ref}=1$. 

In [ ]:
plot=foresee.get_llp_spectrum(mass=mass, coupling=1.0, do_plot=True, save_file=True)
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER during LHC Run 3. 

In [ ]:
foresee.set_detector(
    distance=480, 
    selection="np.sqrt(x.x**2 + x.y**2)<.1", 
    length=1.5, 
    luminosity=250,
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames = ['POWHEG-central', 'POWHEG-max', 'POWHEG-min']

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=None,
)


for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),9))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
from tqdm.notebook import tqdm

masses = np.logspace(-1,np.log10(6.0),50)
couplings = np.logspace(-5,0,50)

# Use cached LLP spectra: get_llp_spectrum recomputes on every call, 
# so skip any masses already saved in model/LLP_spectra/.
for mass in tqdm(masses):
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function.

In [ ]:
# Group production channels by parent hadron 
productions = [
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["3body_pseudo_421","3body_pseudo_-421","3body_vector_421","3body_vector_-421"])], "color": "deepskyblue", "label": r"$D^0$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["2body_411","2body_-411","3body_pseudo_411","3body_pseudo_-411","3body_vector_411","3body_vector_-411"])], "color": "blue", "label": r"$D^\pm$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["2body_431","2body_-431","3body_pseudo_431","3body_pseudo_-431","3body_vector_431","3body_vector_-431"])], "color": "dodgerblue", "label": r"$D_s^\pm$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["2body_521","2body_-521","3body_pseudo_521","3body_pseudo_-521","3body_vector_521","3body_vector_-521"])], "color": "purple", "label": r"$B^\pm$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["3body_pseudo_511","3body_pseudo_-511","3body_vector_511","3body_vector_-511"])], "color": "magenta", "label": r"$B^0$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["3body_pseudo_531","3body_pseudo_-531","3body_vector_531","3body_vector_-531"])], "color": "violet", "label": r"$B_s^0$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["2body_541","2body_-541","3body_pseudo_541","3body_pseudo_-541","3body_vector_541","3body_vector_-541"])], "color": "mediumpurple", "label": r"$B_c^\pm$", "generators": generators_heavy},
    {"channels": [c[0] for c in production_channels if any(s in c[0] for s in ["2body_tau_15","2body_tau_-15","3body_tau_15","3body_tau_-15"])], "color": "green", "label": r"$\tau^\pm$", "generators": generators_heavy},
]

branchings = [
    [('nu', 'nu', 'nu')       , "black"         , "solid" , r"$\nu \nu \nu$"         , .17  , 0.06  ],
    [('nu', 'e', 'anti_e')    , "red"           , "solid" , r"$\nu e^+ e^-$"         , .17  , 0.02  ],
    [('nu', 'mu', 'anti_mu')  , "orange"        , "solid" , r"$\nu \mu^+ \mu^-$"     , 1.1  , 2e-2  ],
    [('nu', 'pi0')            , "blue"          , "solid" , r"$\nu \pi^0$"           , .18  , 0.45  ],
    [('mu', 'anti_tau', 'nu') , "purple"        , "solid" , r"$\nu \mu^\pm \tau^\mp$", 6.1  , 0.02  ],
    [('tau', 'anti_mu', 'nu') , "purple"        , "None"  , None                     , 5.0  , 0.025 ],
    [('tau', 'rho+')          , "brown"         , "solid" , r"$\tau^\mp \rho^\pm $"  , 2 , 2.5e-2],
    [('anti_tau', 'anti_rho+'), "brown"         , "None"  , None                     , .2   , 0.13  ],
    [('nu', 'rho0')           , "teal"          , "solid" , r"$\nu \rho^0 $"         , 1.1  , 5.4e-2],
    
]
branchingsother=['gray', 'solid', 'other', 2, 0.6, [0.1,10]]


plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",
    xlims=[0.1,10],ylims=[4e-4,4e11],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/\varepsilon^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(1.01,1),
    fs_label=12,
    fs_label_br=9,
    ncol=3,
    branchings=branchings,
    branchingsother=branchingsother,
    figsize=(7,6)
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")
plot.show()

Let us now scan over various masses and couplings, and record the resulting number of events. Here we are iterating over various detector configurations: FASER Run 3, FASER HL-LHC, and FASER2 HL-LHC. 

In [ ]:
visible_modes = [mode for mode in list(model.br_finalstate.keys()) if mode != ('nu', 'nu', 'nu') ] 
setupnames = ['POWHEG-central']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_POWHEG-central.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    # ["14TeV_FASER_HL_POWHEG-central.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    # ["14TeV_FASER2_HL_POWHEG-central.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds. 

In [ ]:
bounds = [
        ['bounds_001/bounds_cosmo.txt'        , 'BBN'               ,1.1e-1    ,0.00264     ,-20 ],
        ['bounds_001/bounds_bebc.txt'         , 'BEBC'              , 1        ,0.000775    ,0   ],
        ['bounds_001/bounds_charm.txt'        , 'CHARM'             , 1.27e-1   ,0.00632*1.8 , -25],
        ['bounds_001/bounds_delphi_long.txt'  , 'Delphi \n (long)'  , 2        ,0.007    ,0   ],
        ['bounds_001/bounds_delphi_short.txt' , 'Delphi \n (short)' , 4    , 0.01      ,0 ],
        ['bounds_001/bounds_babar.txt'        , 'BaBar'             , 1.34*.7  ,0.00224     ,-85  ],
        ['bounds_001/bounds_argoneut.txt'     , 'Argoneut'          , 3.5e-1   ,0.0224*1.3  ,-49 ]
        ]

Finally, we can plot everything using `foresee.plot_reach()`. The function `hnl.get_bounds()` is used to import existing constraints.  

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds= bounds,
    projections=[],
    title="HNL's",
    xlims = [0.1, 10], 
    ylims = [1e-4, 1e-1],
    xlabel=r"$m_{N}$ (GeV)", 
    ylabel=r"$\epsilon$",
    legendloc=(1,0.2),
    linewidths=2
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()